# Хирон: QLoRA на бесплатном T4

Ноль оплаты. Runtime → T4 GPU. По умолчанию учится Qwen3-1.7B: его Q4_K_M реально отдавать с Oracle Always Free (оценка 10–18 ток/с). 4B на T4 учится, но на 4 ядрах ARM отвечает примерно 4–8 ток/с. Датасет уже в репозитории, каталог приложения не заменяется.

Оценки скорости — не замер этого ноутбука, а порядок величины для Ampere A1 и бесплатного CPU Space. Подробности: `tools/llm/README.md`.


In [ ]:
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет GPU")


In [ ]:
%%capture
!pip install --upgrade unsloth


In [ ]:
!git clone --depth 1 https://github.com/argunov250-del/bio-age.git
TRAIN = "bio-age/tools/llm/data/train.jsonl"
VAL = "bio-age/tools/llm/data/val.jsonl"


In [ ]:
import os, torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# По умолчанию 1.7B: на бесплатном ARM отвечает за несколько секунд.
# unsloth/Qwen3-0.6B — ещё быстрее. unsloth/Qwen3-4B влезает в T4,
# но на Always Free Oracle отвечает десятки секунд. Если ядро пишет OOM — поставьте 1.7B.
MODEL = os.environ.get("CHIRON_MODEL", "unsloth/Qwen3-1.7B")
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

def to_text(ex):
    kw = dict(tokenize=False, add_generation_prompt=False)
    try:
        text = tokenizer.apply_chat_template(ex["messages"], enable_thinking=False, **kw)
    except TypeError:
        text = tokenizer.apply_chat_template(ex["messages"], **kw)
    return {"text": text}

data = load_dataset("json", data_files={"train": TRAIN, "validation": VAL})
data = data.map(to_text)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        output_dir="out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="no",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        report_to="none",
    ),
)
trainer.train()


In [ ]:
# Слитая модель и GGUF Q4_K_M. Выгрузка только если вы сами задали HF_TOKEN и HF_REPO.
# Ноутбук аккаунт не создаёт.
model.save_pretrained_merged("chiron-merged", tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf("chiron-gguf", tokenizer, quantization_method="q4_k_m")

import os
token = os.environ.get("HF_TOKEN")
repo = os.environ.get("HF_REPO")  # например "my-user/chiron-qwen3"
if token and repo:
    model.push_to_hub_gguf(repo, tokenizer, quantization_method="q4_k_m", token=token)
    print("отправлено", repo)
else:
    print("HF_TOKEN или HF_REPO пусты — GGUF лежит в chiron-gguf/, на Hugging Face ничего не ушло.")
